In [19]:
import numpy as np
import json
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [8]:
def generate_data_set(params):
    inputs_, outputs_ = make_multilabel_classification(
        n_samples=params["sample_size"] + 1,
        n_classes=params["label_number"],
        n_labels=params["avg_label_number"],
        allow_unlabeled=False,
        return_indicator=True,
    )
    return inputs_, outputs_

In [33]:
def resample_data_set(inputs_, outputs_, params):
    sample_size = inputs_.shape[0] - 1
    indices = np.arange(sample_size + 1)
    np.random.shuffle(indices)

    inputs_ = inputs_[indices, :]
    outputs_ = outputs_[indices, :]

    inputs, outputs = (inputs_[:-1, :], outputs_[:-1, :])
    input_test, output_test = (
        inputs_[-1, :].reshape(1, -1),
        outputs_[-1, :].reshape(1, -1),
    )

    inputs_train, inputs_calibration, outputs_train, outputs_calibration = (
        train_test_split(inputs, outputs, test_size=params["cal_size"])
    )

    input_scaler = StandardScaler()
    scaled_inputs_train = input_scaler.fit_transform(inputs_train)
    scaled_inputs_calibration = input_scaler.transform(inputs_calibration)
    scaled_input_test = input_scaler.transform(input_test)

    (
        scaled_inputs_selection,
        scaled_inputs_proper_cal,
        outputs_selection,
        outputs_proper_cal,
    ) = train_test_split(
        scaled_inputs_calibration,
        outputs_calibration,
        test_size=params["proper_cal_size"],
    )

    return (
        (scaled_inputs_train, outputs_train),
        (scaled_inputs_calibration, outputs_calibration),
        (
            scaled_inputs_selection,
            scaled_inputs_proper_cal,
            outputs_selection,
            outputs_proper_cal,
        ),
        (scaled_input_test, output_test),
    )

In [25]:
def reader(path_params):
    with open(path_params, "r") as file:
        params = json.load(file)
    print(params)
    return params

In [26]:
params_global = reader("params/synthetic.json")

{'data': {'sample_size': 1000, 'label_number': 10, 'avg_label_number': 4, 'cal_size': 0.5, 'proper_cal_size': 0.5}}


In [27]:
inputs_, outputs_ = generate_data_set(params_global["data"])

In [34]:
(
    (scaled_inputs_train, outputs_train),
    (scaled_inputs_calibration, outputs_calibration),
    (
        scaled_inputs_selection,
        scaled_inputs_proper_cal,
        outputs_selection,
        outputs_proper_cal,
    ),
    (scaled_input_test, output_test),
) = resample_data_set(inputs_, outputs_, params_global["data"])
print(scaled_inputs_train.shape)
print(outputs_train.shape)

print(scaled_inputs_calibration.shape)
print(outputs_calibration.shape)

print(scaled_inputs_selection.shape)
print(outputs_selection.shape)

print(scaled_inputs_proper_cal.shape)
print(outputs_proper_cal.shape)

print(scaled_input_test.shape)
print(output_test.shape)

(500, 20)
(500, 10)
(500, 20)
(500, 10)
(250, 20)
(250, 10)
(250, 20)
(250, 10)
(1, 20)
(1, 10)
